# NetSentinel AI — nDPI-Aware Network Threat Detection Engine

**Goal:** train a tabular ML detector on network-flow telemetry while treating nDPI application/risk output as first-class features.

Pipeline:

`nDPI normalized flows → feature engineering → leakage-safe split → Random Forest → evaluation → artifact → inference → nDPI + ML risk fusion`

> The notebook supports real labelled CSV/Parquet datasets. A small synthetic dataset is included only so the complete pipeline can be smoke-tested without downloading a dataset.


In [ ]:
# 0. Install (run once if needed)
# %pip install pandas numpy scikit-learn joblib matplotlib seaborn pyarrow

from pathlib import Path
import json, re, math, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
import joblib

ROOT = Path("..")
MODEL_DIR = ROOT / "models"
DATA_DIR = ROOT / "data"
MODEL_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)


In [ ]:
# 1. Canonical nDPI-aware feature builder

NDPI_RISK_WEIGHTS = {
    "RISKY_DOMAIN": 28, "MALICIOUS_HOST": 35, "SUSPICIOUS_DNS": 28,
    "UNUSUAL_PORT": 18, "RISKY": 25
}

def norm_risk(x):
    return re.sub(r"[^A-Z0-9]+", "_", str(x).upper()).strip("_")

def make_features(df):
    d = df.copy()
    numeric = ["packets","bytes","duration_seconds","source_port","destination_port",
               "avg_query_length","dns_query_entropy","outbound_ratio",
               "unique_destinations","failed_connections","syn_packets","rst_packets"]
    for c in numeric:
        if c not in d: d[c] = 0
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)

    d["duration_seconds"] = d["duration_seconds"].clip(lower=0.001)
    d["packets_per_second"] = d["packets"] / d["duration_seconds"]
    d["bytes_per_second"] = d["bytes"] / d["duration_seconds"]
    d["avg_packet_size"] = d["bytes"] / d["packets"].clip(lower=1)

    # nDPI risk list can be represented as semicolon/comma separated text.
    risk_col = d["ndpi_risks"] if "ndpi_risks" in d else pd.Series("", index=d.index)
    risk_text = risk_col.fillna("").astype(str)
    d["ndpi_risk_count"] = risk_text.apply(
        lambda x: 0 if not x else len([v for v in re.split(r"[,;|]", x) if v.strip()])
    )
    d["ndpi_risk_weight"] = risk_text.apply(
        lambda x: sum(NDPI_RISK_WEIGHTS.get(norm_risk(v), 25)
                      for v in re.split(r"[,;|]", x) if v.strip())
    )

    app = d["application"].fillna("UNKNOWN").astype(str).str.upper() if "application" in d else pd.Series("UNKNOWN", index=d.index)
    transport = d["transport"].fillna("UNKNOWN").astype(str).str.upper() if "transport" in d else pd.Series("UNKNOWN", index=d.index)

    for a in ["DNS","HTTP","HTTPS","QUIC","SSH","UNKNOWN","SMTP","FTP","RDP","SMB","TOR","BITTORRENT"]:
        d[f"app_is_{a.lower()}"] = (app == a).astype(int)
    for t in ["TCP","UDP","ICMP","SCTP"]:
        d[f"transport_is_{t.lower()}"] = (transport == t).astype(int)

    d["port_is_dns"] = (d["destination_port"] == 53).astype(int)
    d["port_is_http"] = (d["destination_port"] == 80).astype(int)
    d["port_is_https"] = (d["destination_port"] == 443).astype(int)
    d["port_is_ssh"] = (d["destination_port"] == 22).astype(int)
    d["port_is_telnet"] = d["destination_port"].isin([23,2323]).astype(int)

    for c in ["repeated_destination","high_outbound_ratio"]:
        if c in d:
            d[c] = d[c].astype(str).str.lower().isin(["true","1","yes"]).astype(int)
        else:
            d[c] = 0

    feature_cols = [
        c for c in d.columns
        if c in numeric or c.startswith(("packets_per_","bytes_per_","avg_packet_",
                                         "app_is_","transport_is_","port_is_"))
        or c in ["ndpi_risk_count","ndpi_risk_weight","repeated_destination",
                 "high_outbound_ratio"]
    ]
    return d[feature_cols].replace([np.inf,-np.inf],0).fillna(0), feature_cols


In [ ]:
# 2. Load your real labelled dataset
# Expected: one row per network flow. You need a target column such as Label/label/attack_category.
#
# Example:
# REAL_DATA = DATA_DIR / "flows.csv"
# df = pd.read_csv(REAL_DATA)
#
# Minimum useful columns:
# packets, bytes, duration_seconds, source_port, destination_port,
# transport, application, ndpi_risks, plus a label column.
#
# The engine can also consume common CIC-IDS style names after you map them.

def find_label_column(columns):
    for c in ["label","Label","attack_category","Attack","attack","class","Class"]:
        if c in columns: return c
    return None

# Set this to your dataset path.
REAL_DATA = None

if REAL_DATA:
    df = pd.read_csv(REAL_DATA)
    label_col = find_label_column(df.columns)
    if not label_col:
        raise ValueError("No label column found. Set label_col manually.")
    print(df.shape, df[label_col].value_counts().head(20))
else:
    df = None
    print("No real dataset selected. Next cell creates a smoke-test dataset.")


In [ ]:
# 3. Synthetic smoke-test data — NOT for reporting benchmark accuracy
rng = np.random.default_rng(42)

def synth(n=5000):
    labels = rng.choice(
        ["BENIGN","DNS_TUNNELING","PORT_SCAN","DOS","BOTNET","BRUTE_FORCE","DATA_EXFILTRATION"],
        size=n, p=[.60,.08,.08,.07,.06,.06,.05]
    )
    rows=[]
    for y in labels:
        app = rng.choice(["HTTPS","DNS","HTTP","SSH","UNKNOWN"])
        packets = rng.integers(5,250)
        bytes_ = rng.integers(500,100000)
        duration = rng.uniform(.5,120)
        qlen=0; risks=""
        repeated=False; outbound=False
        dst=rng.choice([53,80,443,22,23,2323,random_port:=rng.integers(1024,60000)])
        if y=="DNS_TUNNELING":
            app="DNS"; dst=53; packets=rng.integers(250,1000); qlen=rng.integers(60,150)
            risks="SUSPICIOUS_DNS"; repeated=True
        elif y=="PORT_SCAN":
            app="UNKNOWN"; packets=rng.integers(100,500); dst=rng.integers(1,1024)
        elif y=="DOS":
            packets=rng.integers(5000,50000); duration=rng.uniform(1,30)
        elif y=="DATA_EXFILTRATION":
            bytes_=rng.integers(5_000_000,50_000_000); outbound=True; repeated=True
        elif y=="BRUTE_FORCE":
            app="SSH"; dst=22; packets=rng.integers(100,800); risks="UNUSUAL_PORT"
        elif y=="BOTNET":
            risks="RISKY_DOMAIN"; repeated=True
        rows.append(dict(
            packets=packets, bytes=bytes_, duration_seconds=duration,
            source_port=rng.integers(1024,65535), destination_port=dst,
            transport="UDP" if app=="DNS" else "TCP", application=app,
            ndpi_risks=risks, avg_query_length=qlen,
            dns_query_entropy=rng.uniform(0,5) if app=="DNS" else 0,
            outbound_ratio=.9 if outbound else rng.uniform(.05,.5),
            unique_destinations=rng.integers(30,200) if y=="PORT_SCAN" else rng.integers(1,8),
            failed_connections=rng.integers(10,100) if y in ["PORT_SCAN","BRUTE_FORCE"] else 0,
            syn_packets=rng.integers(30,300) if y=="PORT_SCAN" else 0,
            rst_packets=rng.integers(5,50) if y=="PORT_SCAN" else 0,
            repeated_destination=repeated, high_outbound_ratio=outbound, label=y
        ))
    return pd.DataFrame(rows)

if df is None:
    df=synth()
    label_col="label"
print(df.shape)
display(df.head())


In [ ]:
# 4. Train/evaluate
# IMPORTANT: For real datasets, remove duplicate rows and prefer a host/time-aware split
# when the same source appears in both train and test.

df = df.dropna(subset=[label_col]).copy()
df[label_col] = df[label_col].astype(str).str.upper().str.replace(r"[^A-Z0-9_]+","_",regex=True)

X, feature_names = make_features(df)
y_raw = df[label_col]

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=350,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Balanced accuracy:", round(balanced_accuracy_score(y_test,pred),4))
print("Macro F1:", round(f1_score(y_test,pred,average="macro"),4))
print(classification_report(y_test,pred,target_names=encoder.classes_,zero_division=0))


In [ ]:
# 5. Feature importance — useful for explaining what the model learned
imp = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
display(imp.head(20).to_frame("importance"))


In [ ]:
# 6. Save production artifact
artifact = {
    "model": model,
    "label_encoder": encoder,
    "feature_names": feature_names,
    "feature_version": "ndpi-flow-v1",
    "classes": list(encoder.classes_),
}
artifact_path = MODEL_DIR / "ndpi_detector.joblib"
joblib.dump(artifact, artifact_path)

schema = {
    "feature_version":"ndpi-flow-v1",
    "feature_names":feature_names,
    "classes":list(encoder.classes_),
    "notes":"nDPI-aware network flow classifier; nDPI risks are included as explicit features."
}
(DATA_DIR / "ml_feature_schema.json").write_text(json.dumps(schema,indent=2))
print("Saved:", artifact_path)


In [ ]:
# 7. Inference test using the same flow schema that the backend receives
import sys
sys.path.append(str(ROOT))
from ml.detection_engine import DetectionEngine

engine = DetectionEngine(artifact_path)
sample = {
    "flow_id":"NOTEBOOK-001",
    "packets":380, "bytes":42000, "duration_seconds":60,
    "source_port":53001, "destination_port":53,
    "transport":"UDP", "application":"DNS",
    "ndpi_risks":["SUSPICIOUS_DNS"],
    "metadata":{"avg_query_length":72,"repeated_destination":True}
}
result = engine.predict(sample)
result


## Production notes

1. **Do not train on raw IP addresses, flow IDs, timestamps, or payload text** unless you have a very specific leakage-safe reason.
2. If a dataset has many flows from the same host/session, a random split can inflate metrics. Prefer grouping by source host, capture, or time for serious evaluation.
3. nDPI is the deterministic DPI/risk layer; this model is a behavioural classifier. The backend fuses both signals.
4. The synthetic dataset is for pipeline verification only.
5. Keep raw packet payloads out of the AI explanation service. Send normalized metadata/evidence instead.
